In [303]:
import numpy as np
import pandas as pd
import os
import sys
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import scipy.stats as stats 
import missingno as msno
import sklearn as sk
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler, PowerTransformer
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from feature_engine.outliers import Winsorizer
sys.path.append("../src/Preprocesamiento")
from custom_transformers import OutlierHandler
import matplotlib.pyplot as plt
import plotly

In [304]:
SEMILLA = 42
FILENAME = "dataset_final.csv"
datapath = os.path.join("..","data","processed",FILENAME)
df = pd.read_csv(datapath)

In [305]:
y = df['target_Frio_kW_next']
df = df.drop(columns=['target_Frio_kW_next'])

In [306]:
with open ("../src/Preprocesamiento/preprocesamiento.pkl", "rb") as f:
    preprocessor = pickle.load(f)

In [307]:
df_preprocesado = preprocessor.transform(df)


### Separación de datos

In [308]:
X_train, X_test, y_train, y_test = train_test_split(df, y, train_size=0.7, shuffle=False, random_state=SEMILLA)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, train_size=0.5, shuffle=False, random_state=SEMILLA)


In [309]:
X_train = preprocessor.transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

### Baselines: MAE

Día de hoy

In [310]:
print(type(preprocessor.named_steps['Escalado']))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [311]:
columns_no_ciclicas = X_train.columns[~X_train.columns.isin(["cos_month_of_year", "sin_month_of_year"])]
df_sin_escalar = (
    preprocessor.named_steps['Escalado']
    .named_transformers_['Escalado']
    .set_output(transform="pandas")
    .inverse_transform(X_val[columns_no_ciclicas])
)
df_sin_escalar = pd.DataFrame(df_sin_escalar, columns=columns_no_ciclicas)

In [312]:
y_val

805    16144.0
806    12401.0
807    15177.0
808    17596.0
809    17489.0
        ...   
972    20378.0
973    21941.0
974    22115.0
975    17634.0
976    22816.0
Name: target_Frio_kW_next, Length: 172, dtype: float64

In [313]:
y_frio_hoy = df_sin_escalar['Frio (Kw)'].values
MAE_frio_hoy = np.mean(np.abs(y_frio_hoy - y_val))
print(f"MAE entre Frio (Kw) y target_Frio_kW_next: {MAE_frio_hoy}")

MAE entre Frio (Kw) y target_Frio_kW_next: 2724.4767441860463


Promedio móvil

In [314]:
y_prom_movil = df_sin_escalar['Prom movil Frio'].values
MAE_prom_movil = np.mean(np.abs(y_prom_movil - y_val))
print(f"MAE entre Prom movil Frio y target_Frio_kW_next: {MAE_prom_movil:.4f}")

MAE entre Prom movil Frio y target_Frio_kW_next: 2393.5963


## Entrenamiento de modelos

### GXBoost

In [315]:
gxboost = GradientBoostingRegressor(loss="absolute_error" , random_state=SEMILLA)
gxboost.fit(X_train, y_train)
y_pred = gxboost.predict(X_val)
MAE_gxboost = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 2179.6897


### Random Forest Regressor

In [316]:
randomforest = RandomForestRegressor(criterion="absolute_error",random_state=SEMILLA)
randomforest.fit(X_train, y_train)
MAE_randomforest = np.mean(np.abs(randomforest.predict(X_val) - y_val))
print(f"MAE: {MAE_randomforest:.4f}")

MAE: 2319.4289


### lightGBM

In [317]:
lightgbm = LGBMRegressor(objective="mae",random_state=SEMILLA,verbose=-1)
lightgbm.fit(X_train, y_train)
y_pred = lightgbm.predict(X_val)
MAE_lightgbm = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lightgbm:.4f}")

MAE: 2289.4630


### Lasso

In [318]:
lasso = Lasso(random_state=SEMILLA)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_val)
MAE_lasso = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lasso:.4f}")

MAE: 2359.1529


### Busqueda de hiperparámetros

### GXBoost

In [326]:
def objective_gxb(trial):
    
   n_estimators = trial.suggest_int("n_estimators", 50,500)
   max_depth = trial.suggest_int("max_depth", 1, 5)
   min_samples_split = trial.suggest_int("min_samples_split", 2,30)
   min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 30)

   learning_rate = trial.suggest_float("learning_rate", 0.005, 0.15, log=True)
   
   
   model = GradientBoostingRegressor(
       loss="absolute_error",
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       min_samples_leaf= min_samples_leaf,
       learning_rate= learning_rate,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score




sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction = "minimize", sampler=sampler)
study.optimize(objective_gxb, n_trials=100, n_jobs=1)

[I 2026-11-14 18:46:50,297] A new study created in memory with name: no-name-7984a9d9-741e-496b-8518-751c7a190d80
[I 2026-11-14 18:46:51,512] Trial 0 finished with value: 2397.6882666614947 and parameters: {'n_estimators': 218, 'max_depth': 5, 'min_samples_split': 23, 'min_samples_leaf': 18, 'learning_rate': 0.00850018649460551}. Best is trial 0 with value: 2397.6882666614947.
[I 2026-11-14 18:46:51,813] Trial 1 finished with value: 2236.4475749269754 and parameters: {'n_estimators': 120, 'max_depth': 1, 'min_samples_split': 27, 'min_samples_leaf': 19, 'learning_rate': 0.05557494721547488}. Best is trial 1 with value: 2236.4475749269754.
[I 2026-11-14 18:46:52,164] Trial 2 finished with value: 4662.945103429732 and parameters: {'n_estimators': 59, 'max_depth': 5, 'min_samples_split': 26, 'min_samples_leaf': 7, 'learning_rate': 0.009279990423245293}. Best is trial 1 with value: 2236.4475749269754.
[I 2026-11-14 18:46:52,501] Trial 3 finished with value: 2518.1639709350666 and parameters

In [328]:
X_cool = pd.concat([X_train,X_val] , axis=0)
y_cool = pd.concat([y_train,y_val], axis=0)
gxboost = GradientBoostingRegressor( **study.best_params ,random_state=SEMILLA)
gxboost.fit(X_cool, y_cool)
y_pred = gxboost.predict(X_test)
MAE_gxboost = np.mean(np.abs(y_pred - y_test))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 2780.5488


In [94]:
import optuna.visualization as vis
import plotly

In [321]:

fig_history = vis.plot_optimization_history(study)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

### RandomForest

In [322]:
def objective_rf(trial):
   n_estimators = trial.suggest_int("n_estimators", 50, 300)
   max_depth = trial.suggest_int("max_depth", 5, 30)
   min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
   min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

   model = RandomForestRegressor(
      criterion="absolute_error",
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       min_samples_leaf= min_samples_leaf,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study_rf = optuna.create_study(direction = "minimize", sampler=sampler)
study_rf.optimize(objective_rf, n_trials=100)

[I 2026-11-14 18:36:49,512] A new study created in memory with name: no-name-aa7f09c0-86fc-4f72-9c39-3332caa0a1ca
[I 2026-11-14 18:36:51,143] Trial 0 finished with value: 2258.902052681538 and parameters: {'n_estimators': 147, 'max_depth': 12, 'min_samples_split': 17, 'min_samples_leaf': 4}. Best is trial 0 with value: 2258.902052681538.
[I 2026-11-14 18:36:52,320] Trial 1 finished with value: 2275.6845445736435 and parameters: {'n_estimators': 120, 'max_depth': 19, 'min_samples_split': 4, 'min_samples_leaf': 9}. Best is trial 0 with value: 2258.902052681538.
[I 2026-11-14 18:36:53,130] Trial 2 finished with value: 2341.5074384404925 and parameters: {'n_estimators': 68, 'max_depth': 30, 'min_samples_split': 16, 'min_samples_leaf': 2}. Best is trial 0 with value: 2258.902052681538.
[I 2026-11-14 18:36:53,657] Trial 3 finished with value: 2328.901447788418 and parameters: {'n_estimators': 51, 'max_depth': 26, 'min_samples_split': 15, 'min_samples_leaf': 8}. Best is trial 0 with value: 22

In [185]:

fig_history = vis.plot_optimization_history(study_rf)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_rf)
fig_history.show()
fig_contour = vis.plot_contour(study_rf)
fig_contour.show()


fig_importance = vis.plot_param_importances(study_rf)
fig_importance.show()

### lightGBM

In [329]:
def objective_lgbm(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,200)
   max_depth = trial.suggest_int("max_depth", 3, 20)
   learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
   reg_alpha = trial.suggest_float("reg_alpha", 0.0, 0.1)
   reg_lambda = trial.suggest_float("reg_lambda", 0.0, 0.1)
   
   model = LGBMRegressor(
       n_estimators= n_estimators,
       max_depth= max_depth,
       learning_rate= learning_rate,
       reg_alpha= reg_alpha,
       reg_lambda= reg_lambda,
       random_state=SEMILLA,
       verbose=-1
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study_lgbm = optuna.create_study(direction = "minimize", sampler=sampler)
study_lgbm.optimize(objective_lgbm, n_trials=100, n_jobs=1)

[I 2026-11-14 18:49:29,215] A new study created in memory with name: no-name-6d43d3e0-9827-4cc5-8497-3f61cf6615d5


[I 2026-11-14 18:49:29,436] Trial 0 finished with value: 2719.5861902770225 and parameters: {'n_estimators': 196, 'max_depth': 16, 'learning_rate': 0.2824546930536148, 'reg_alpha': 0.08948273504276488, 'reg_lambda': 0.05978999788110852}. Best is trial 0 with value: 2719.5861902770225.
[I 2026-11-14 18:49:29,495] Trial 1 finished with value: 2493.76194383592 and parameters: {'n_estimators': 189, 'max_depth': 4, 'learning_rate': 0.0668350301015521, 'reg_alpha': 0.004522728891053806, 'reg_lambda': 0.032533033076326434}. Best is trial 1 with value: 2493.76194383592.
[I 2026-11-14 18:49:29,567] Trial 2 finished with value: 2688.925162945661 and parameters: {'n_estimators': 108, 'max_depth': 7, 'learning_rate': 0.2503338776540595, 'reg_alpha': 0.035675332669358926, 'reg_lambda': 0.02809345096873808}. Best is trial 1 with value: 2493.76194383592.
[I 2026-11-14 18:49:29,636] Trial 3 finished with value: 2606.9976838987222 and parameters: {'n_estimators': 131, 'max_depth': 5, 'learning_rate': 0

In [331]:
X_cool = pd.concat([X_train,X_val] , axis=0)
y_cool = pd.concat([y_train,y_val], axis=0)
gxboost = LGBMRegressor( **study_lgbm.best_params ,random_state=SEMILLA)
gxboost.fit(X_cool, y_cool)
y_pred = gxboost.predict(X_test)
MAE_gxboost = np.mean(np.abs(y_pred - y_test))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 2910.8398


In [330]:

fig_history = vis.plot_optimization_history(study_lgbm)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_lgbm)
fig_history.show()
fig_contour = vis.plot_contour(study_lgbm)
fig_contour.show()


fig_importance = vis.plot_param_importances(study_lgbm)
fig_importance.show()

### Lasso

In [332]:
def objective_lasso(trial):
   alpha = trial.suggest_float("alpha", 0.1, 5000, log=True)
   
   model = Lasso(
       alpha= alpha,
       random_state=SEMILLA,
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study_lasso = optuna.create_study(direction = "minimize")
study_lasso.optimize(objective_lasso, n_trials=100)

[I 2026-11-14 18:50:34,761] A new study created in memory with name: no-name-b8806dab-955c-4770-99a8-a235e5cbe221
[I 2026-11-14 18:50:34,780] Trial 0 finished with value: 2360.3286562590583 and parameters: {'alpha': 4.243456233956033}. Best is trial 0 with value: 2360.3286562590583.
[I 2026-11-14 18:50:34,789] Trial 1 finished with value: 2266.6203267175465 and parameters: {'alpha': 107.76907665157752}. Best is trial 1 with value: 2266.6203267175465.
[I 2026-11-14 18:50:34,799] Trial 2 finished with value: 3114.935919209667 and parameters: {'alpha': 1582.6311117259577}. Best is trial 1 with value: 2266.6203267175465.
[I 2026-11-14 18:50:34,806] Trial 3 finished with value: 2333.976198455772 and parameters: {'alpha': 23.17190488078028}. Best is trial 1 with value: 2266.6203267175465.
[I 2026-11-14 18:50:34,814] Trial 4 finished with value: 2297.0072953836243 and parameters: {'alpha': 485.7920493964195}. Best is trial 1 with value: 2266.6203267175465.
[I 2026-11-14 18:50:34,818] Trial 5 

In [333]:

fig_history = vis.plot_optimization_history(study_lasso)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_lasso)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

In [334]:
X_cool = pd.concat([X_train,X_val] , axis=0)
y_cool = pd.concat([y_train,y_val], axis=0)
gxboost = Lasso( **study_lasso.best_params ,random_state=SEMILLA)
gxboost.fit(X_cool, y_cool)
y_pred = gxboost.predict(X_test)
MAE_gxboost = np.mean(np.abs(y_pred - y_test))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 2893.8246


In [ ]:
def objective_gxboost(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 3, 9)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2, log=True)
    
    # Regularización y muestreo para evitar overfitting
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    subsample = trial.suggest_float("subsample", 0.6, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)
    
    reg_alpha = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True)

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=SEMILLA,
        eval_metric="mae"   
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    score = np.mean(np.abs(y_pred - y_val))
    return score

study_gxboost = optuna.create_study(direction="minimize")
study_gxboost.optimize(objective_gxboost, n_trials=100)

[I 2026-11-14 18:51:17,142] A new study created in memory with name: no-name-ad961759-2914-4748-a261-d35aa12d062b
[I 2026-11-14 18:51:17,644] Trial 0 finished with value: 2533.3984999545787 and parameters: {'n_estimators': 267, 'max_depth': 9, 'learning_rate': 0.04197334948337011, 'min_child_weight': 8, 'subsample': 0.8673533503368058, 'colsample_bytree': 0.6318412855882869, 'reg_alpha': 0.0040669265589025495, 'reg_lambda': 0.059130875052410624}. Best is trial 0 with value: 2533.3984999545787.
[I 2026-11-14 18:51:18,201] Trial 1 finished with value: 2880.011880564135 and parameters: {'n_estimators': 421, 'max_depth': 6, 'learning_rate': 0.19236638367561082, 'min_child_weight': 8, 'subsample': 0.6203228742327113, 'colsample_bytree': 0.6754781439862988, 'reg_alpha': 0.030639018883172567, 'reg_lambda': 0.24588262869650174}. Best is trial 0 with value: 2533.3984999545787.
[I 2026-11-14 18:51:18,485] Trial 2 finished with value: 2799.5514285065406 and parameters: {'n_estimators': 201, 'max_